# Tratamento das bases originais

# Imports 

In [2]:
import pandas as pd
import datetime
from datetime import datetime, date 


# 2017 pre processing

## Selecionando colunas, concat e to_parquet

In [163]:


colunas = ['NU_ANO','SG_UF_IES','SG_UF_CAMPUS','NO_MUNUCIPIO_CAMPUS', 'NO_CURSO',
                                    'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA','NU_CPF', 
                                    'NO_INSCRITO', 'TP_SEXO','DT_NASCIMENTO', 'SG_UF_CANDIDATO', 
                                    'MUNICIPIO_CANDIDATO','ST_APROVADO']


sisu_regular_1_2017 = pd.read_csv ('ListagemChamadaRegular_2017-1.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2017 = pd.read_csv ('ListagemChamadaRegular_2017-2.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2017 = pd.read_csv ('ListagemListaEspera_2017-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2017 = pd.read_csv ('ListagemListaEspera_2017-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2017 = pd.concat ([sisu_regular_1_2017,sisu_regular_2_2017,sisu_espera_1_2017,sisu_espera_2_2017])

sisu_concat_2017.to_parquet ('sisu_2017_clean.parquet')

sisu_2017_clean = pd.read_parquet ('sisu_2017_clean.parquet')


In [251]:
sisu_2017_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop an missing values

In [237]:
sisu_2017_clean.drop_duplicates()
sisu_2017_clean.dropna()
sisu_2017_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
ST_APROVADO            0
dtype: int64

## date dealing

- Algumas linhas estavam gerando erro, então optei por 'drop()';
- As datas de nascimento estavam em formato 'object';
- Algumas datas tinham valores não consistentes, p.ex: '31/12/4000'

In [241]:
dfx = sisu_2017_clean.copy()

fuzzy_lines = [1327532,1327435,1327531,1327530,1327529,1327528]
dfx = dfx.drop (fuzzy_lines, axis = 0)

df1 = dfx[dfx['DT_NASCIMENTO'] != '31/12/4000']

df1.head()

sisu_2017_clean = df1.copy()

sisu_2017_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'ST_APROVADO'],
      dtype='object')

## 2017 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [29]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')


pibpc.head(30)

,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [243]:
sisu_2017_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'ST_APROVADO'],
      dtype='object')

In [253]:


sisu_2017_clean = sisu_2017_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'})

sisu_2017_clean ['municipio'] = sisu_2017_clean.municipio.str.lower()

sisu_2017_pib = pd.merge (sisu_2017_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2017_pib_clean = sisu_2017_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
                     
sisu_2017_pib_clean = sisu_2017_pib_clean.dropna()

print ('sisu_2017_pib_clean shape',sisu_2017_pib_clean.shape)
print ('sisu 2017 shape',sisu_2017_clean.shape)

sisu_2017_pib_clean shape (1631137, 17)
sisu 2017 shape (1634820, 14)


In [187]:
sisu_2017_pib_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'municipio', 'ST_APROVADO', 'ano', 'uf', 'pib_pc'],
      dtype='object')

In [255]:
df_all = sisu_2017_pib_clean.copy()

# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df_all['idade'] = df_all['DT_NASCIMENTO'].apply(age)
df_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'municipio', 'ST_APROVADO', 'ano', 'uf', 'pib_pc', 'idade'],
      dtype='object')

In [333]:

sisu_2017_pib_clean = df_all.copy()
sisu_2017_pib_clean.to_parquet('sisu_2017_pib_clean.parquet')
df_all.head()


,NU_ANO,SG_UF_IES,SG_UF_CAMPUS,NO_MUNICIPIO_CAMPUS,NO_CURSO,TP_MOD_CONCORRENCIA,DS_MOD_CONCORRENCIA,NU_CPF,NO_INSCRITO,TP_SEXO,DT_NASCIMENTO,SG_UF_CANDIDATO,municipio,ST_APROVADO,ano,uf,pib_pc,idade
0,2017,SP,SP,São Carlos,PROCESSOS GERENCIAIS,L,"Candidatos que, independentemente da renda (ar...",XXX.158.218-XX,DAILA MARIANA DA SILVA,F,1997-03-03,SP,são carlos,N,2020,SP,47701,27
2,2017,SP,SP,São Carlos,ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,L,Candidatos com deficiência autodeclarados pret...,XXX.901.118-XX,FLAVIO PEREIRA,M,1967-05-19,SP,são carlos,N,2020,SP,47701,57
4,2017,SP,SP,São Carlos,ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,L,"Candidatos autodeclarados pretos, pardos ou in...",XXX.790.038-XX,PHELLIPE BOAZ JARDIM PEDRELLA,M,1993-03-17,SP,são carlos,N,2020,SP,47701,31
6,2017,SP,SP,São Carlos,PROCESSOS GERENCIAIS,L,"Candidatos autodeclarados pretos, pardos ou in...",XXX.790.038-XX,PHELLIPE BOAZ JARDIM PEDRELLA,M,1993-03-17,SP,são carlos,N,2020,SP,47701,31
8,2017,RJ,RJ,Niterói,CINEMA E AUDIOVISUAL,A,Ampla concorrência,XXX.383.558-XX,PEDRO AUGUSTO FATORE,M,1996-08-02,SP,são carlos,N,2020,SP,47701,28


# 2018 pre processing

## Selecionando colunas, concat e to_parquet

In [4]:
import pandas as pd

colunas = ['ANO','UF_IES', 'UF_CAMPUS','MUNICIPIO_CAMPUS','NOME_CURSO','TIPO_MOD_CONCORRENCIA', 'MOD_CONCORRENCIA',
           'CPF', 'INSCRICAO_ENEM', 'INSCRITO', 'SEXO', 'DATA_NASCIMENTO', 'UF_CANDIDATO', 'MUNUCIPIO_CANDIDATO', 
           'APROVADO']

sisu_regular_1_2018 = pd.read_csv ('relatorio_chamada_20_regular_SISU_1_2018.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2018 = pd.read_csv ('relatorio_chamada_20_regular_SISU_2_2018.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2018 = pd.read_csv ('ListagemListaEspera_2018-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2018 = pd.read_csv ('ListagemListaEspera_2018-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2018 = pd.concat ([sisu_regular_1_2018,sisu_regular_2_2018,sisu_espera_1_2018,sisu_espera_2_2018])

sisu_concat_2018.to_parquet ('sisu_2018_clean.parquet')


sisu_2018_clean = pd.read_parquet ('sisu_2018_clean.parquet')


In [371]:
sisu_2018_clean = pd.read_parquet ('sisu_2018_clean.parquet')

sisu_2018_clean.drop (columns = ['ano', 'uf', 'pib_pc'], inplace = True)
sisu_2018_clean.rename (columns = {'municipio':'MUNUCIPIO_CANDIDATO'}, inplace = True)
sisu_2018_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNUCIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop and missing values

In [373]:
sisu_2018_clean.dropna(inplace = True)
sisu_2018_clean.drop_duplicates()
sisu_2018_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
MUNUCIPIO_CANDIDATO    0
ST_APROVADO            0
dtype: int64

## Rename 

In [375]:
sisu_2018_clean.rename (columns = {'ANO':'NU_ANO','UF_IES':'SG_UF_IES', 'UF_CAMPUS':'SG_UF_CAMPUS',
                        'MUNICIPIO_CAMPUS':'NO_MUNICIPIO_CAMPUS','NOME_CURSO':'NO_CURSO',
                        'TIPO_MOD_CONCORRENCIA':'TP_MOD_CONCORRENCIA', 'MOD_CONCORRENCIA':'DS_MOD_CONCORRENCIA',
                        'CPF':'NU_CPF', 'INSCRICAO_ENEM':'CO_INSCRICAO_ENEM', 'INSCRITO':'NO_INSCRITO', 
                        'SEXO':'TP_SEXO', 'DATA_NASCIMENTO':'DT_NASCIMENTO', 
                        'UF_CANDIDATO':'SG_UF_CANDIDATO', 'MUNUCIPIO_CANDIDATO':'MUNICIPIO_CANDIDATO', 
                        'APROVADO':'ST_APROVADO'}, inplace = True)
sisu_2018_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## date dealing


In [377]:
df18_all = sisu_2018_clean.copy()


# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df18_all['idade'] = df18_all['DT_NASCIMENTO'].apply(age)
df18_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO', 'idade'],
      dtype='object')

## 2018 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [67]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')

print ('shape: ', pibpc.shape)
pibpc.head(30)

shape:  (5570, 4)


,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [379]:
sisu_2018_clean = df18_all.copy()
print ('shape: ',sisu_2018_clean.shape)

shape:  (3798746, 16)


In [381]:


sisu_2018_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'}, inplace = True)

sisu_2018_clean ['municipio'] = sisu_2018_clean.municipio.str.lower()

sisu_2018_pib = pd.merge (sisu_2018_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2018_pib_clean = sisu_2018_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
sisu_2018_pib_clean = sisu_2018_pib_clean.dropna()

print ('sisu_2018_pib_clean shape',sisu_2018_pib_clean.shape)
print ('sisu 2018 shape',sisu_2018_clean.shape)
print ('Columns', sisu_2018_pib_clean.columns)

sisu_2018_pib_clean shape (3788871, 19)
sisu 2018 shape (3798746, 16)
Columns Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf',
       'pib_pc'],
      dtype='object')


In [383]:
sisu_2018_pib_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
municipio              0
ST_APROVADO            0
idade                  0
ano                    0
uf                     0
pib_pc                 0
dtype: int64

In [385]:
sisu_2018_pib_clean.to_parquet('sisu_2018_pib_clean.parquet')

# 2019 pre processing

## Selecionando colunas, concat e to_parquet

In [89]:

colunas = ['NU_ANO','SG_UF_IES','SG_UF_CAMPUS','NO_MUNUCIPIO_CAMPUS', 'NO_CURSO',
                                   'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA','NU_CPF', 'CO_INSCRICAO_ENEM',
                                   'NO_INSCRITO', 'TP_SEXO','DT_NASCIMENTO', 'SG_UF_CANDIDATO', 
                                   'MUNICIPIO_CANDIDATO','ST_APROVADO']

sisu_regular_1_2019 = pd.read_csv ('ListagemChamadaRegular_2019.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2019 = pd.read_csv ('ListagemChamadaRegular_2019.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2019 = pd.read_csv ('ListagemListaEspera_2019-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2019 = pd.read_csv ('ListagemListaEspera_2019-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2019 = pd.concat ([sisu_regular_1_2019,sisu_regular_2_2019,sisu_espera_1_2019,sisu_espera_2_2019])

sisu_concat_2019.to_parquet ('sisu_2019_clean.parquet')

sisu_2019_clean = pd.read_parquet ('sisu_2019_clean.parquet')


In [261]:

sisu_2019_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop and missing values

In [263]:
sisu_2019_clean.dropna(inplace = True)
sisu_2019_clean.drop_duplicates()
sisu_2019_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
MUNICIPIO_CANDIDATO    0
ST_APROVADO            0
dtype: int64

## date dealing


In [265]:
df19_all = sisu_2019_clean.copy()

# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df19_all['idade'] = df19_all['DT_NASCIMENTO'].apply(age)
df19_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO', 'idade'],
      dtype='object')

## 2019 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [95]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')

print ('shape: ', pibpc.shape)
pibpc.head(30)

shape:  (5570, 4)


,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [267]:
sisu_2019_clean = df19_all.copy()
print ('shape: ',sisu_2019_clean.shape)

shape:  (3199342, 16)


In [271]:


sisu_2019_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'}, inplace = True)

sisu_2019_clean ['municipio'] = sisu_2019_clean.municipio.str.lower()

sisu_2019_pib = pd.merge (sisu_2019_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2019_pib_clean = sisu_2019_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
sisu_2019_pib_clean = sisu_2019_pib_clean.dropna()

print ('sisu_2019_pib_clean shape',sisu_2019_pib_clean.shape)
print ('sisu 2019 shape',sisu_2019_clean.shape)
print ('columns: ', sisu_2019_pib_clean.columns)

sisu_2019_pib_clean shape (3191671, 19)
sisu 2019 shape (3199342, 16)
columns:  Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf',
       'pib_pc'],
      dtype='object')


In [363]:
sisu_2019_pib_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
municipio              0
ST_APROVADO            0
idade                  0
ano                    0
uf                     0
pib_pc                 0
dtype: int64

In [329]:
sisu_2019_pib_clean.to_parquet('sisu_2019_pib_clean.parquet')

# 2020 pre processing

## Selecionando colunas, concat e to_parquet

In [108]:
colunas = ['NU_ANO','SG_UF_IES','SG_UF_CAMPUS','NO_MUNUCIPIO_CAMPUS', 'NO_CURSO',
                                   'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA','NU_CPF', 'CO_INSCRICAO_ENEM',
                                   'NO_INSCRITO', 'TP_SEXO','DT_NASCIMENTO', 'SG_UF_CANDIDATO', 
                                   'MUNICIPIO_CANDIDATO','ST_APROVADO']

sisu_regular_1_2020 = pd.read_csv ('ListagemChamadaRegular_2020.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2020 = pd.read_csv ('ListagemChamadaRegular_2020.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2020 = pd.read_csv ('ListagemListaEspera_2020-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2020 = pd.read_csv ('ListagemListaEspera_2020-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2020 = pd.concat ([sisu_regular_1_2020,sisu_regular_2_2020,sisu_espera_1_2020,sisu_espera_2_2020])

sisu_concat_2020.to_parquet ('sisu_2020_clean.parquet')


sisu_concat_2020 = pd.read_parquet ('sisu_2020_clean.parquet')


In [276]:
sisu_concat_2020.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop and missing values

In [278]:
sisu_concat_2020.dropna(inplace = True)
sisu_concat_2020.drop_duplicates()
sisu_concat_2020.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
MUNICIPIO_CANDIDATO    0
ST_APROVADO            0
dtype: int64

## date dealing


In [280]:
df20_all = sisu_concat_2020.copy()


# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df20_all['idade'] = df20_all['DT_NASCIMENTO'].apply(age)
df20_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'MUNICIPIO_CANDIDATO', 'ST_APROVADO', 'idade'],
      dtype='object')

## 2020 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [114]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')

print ('shape: ', pibpc.shape)
pibpc.head(30)

shape:  (5570, 4)


,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [284]:
sisu_2020_clean = df20_all.copy()
print ('shape: ',sisu_2020_clean.shape)

shape:  (800038, 16)


In [321]:


sisu_2020_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'}, inplace = True)

sisu_2020_clean ['municipio'] = sisu_2020_clean.municipio.str.lower()

sisu_2020_pib = pd.merge (sisu_2020_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2020_pib_clean = sisu_2020_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
sisu_2020_pib_clean = sisu_2020_pib_clean.dropna()

print ('sisu_2020_pib_clean shape',sisu_2020_pib_clean.shape)
print ('sisu 2020 shape',sisu_2020_clean.shape)
print ('columns: ', sisu_2020_pib_clean.columns)

sisu_2020_pib_clean shape (697695, 19)
sisu 2020 shape (800038, 16)
columns:  Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf',
       'pib_pc'],
      dtype='object')


In [359]:
sisu_2020_pib_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
CO_INSCRICAO_ENEM      0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
municipio              0
ST_APROVADO            0
idade                  0
ano                    0
uf                     0
pib_pc                 0
dtype: int64

In [327]:
sisu_2020_pib_clean.to_parquet('sisu_2020_pib_clean.parquet')

# 2021 pre processing

## Selecionando colunas, concat e to_parquet

In [128]:
colunas = ['NU_ANO','SG_UF_IES','SG_UF_CAMPUS','NO_MUNUCIPIO_CAMPUS', 'NO_CURSO',
                                   'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA','NU_CPF', 'CO_INSCRICAO_ENEM',
                                   'NO_INSCRITO', 'TP_SEXO','DT_NASCIMENTO', 'SG_UF_CANDIDATO', 
                                   'MUNICIPIO_CANDIDATO','ST_APROVADO']

sisu_regular_1_2021 = pd.read_csv ('ListagemChamadaRegular_2021.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2021 = pd.read_csv ('ListagemChamadaRegular_2021.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2021 = pd.read_csv ('ListagemListaEspera_2021-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2021 = pd.read_csv ('ListagemListaEspera_2021-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2021 = pd.concat ([sisu_regular_1_2021,sisu_regular_2_2021,sisu_espera_1_2021,sisu_espera_2_2021])

sisu_concat_2021.to_parquet ('sisu_2021_clean.parquet')


sisu_concat_2021 = pd.read_parquet ('sisu_2021_clean.parquet')


In [291]:

sisu_concat_2021.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop and missing values

In [293]:
sisu_concat_2021.dropna(inplace = True)
sisu_concat_2021.drop_duplicates()
sisu_concat_2021.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
MUNICIPIO_CANDIDATO    0
ST_APROVADO            0
dtype: int64

## date dealing


In [295]:
df21_all = sisu_concat_2021.copy()


# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df21_all['idade'] = df21_all['DT_NASCIMENTO'].apply(age)
df21_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'MUNICIPIO_CANDIDATO', 'ST_APROVADO', 'idade'],
      dtype='object')

## 2021 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [135]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')

print ('shape: ', pibpc.shape)
pibpc.head(30)

shape:  (5570, 4)


,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [299]:
sisu_2021_clean = df21_all.copy()
print ('shape: ',sisu_2021_clean.shape)

shape:  (2357524, 15)


In [319]:


sisu_2021_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'}, inplace = True)

sisu_2021_clean ['municipio'] = sisu_2021_clean.municipio.str.lower()

sisu_2021_pib = pd.merge (sisu_2021_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2021_pib_clean = sisu_2021_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
sisu_2021_pib_clean = sisu_2021_pib_clean.dropna()

print ('sisu_2021_pib_clean shape',sisu_2021_pib_clean.shape)
print ('sisu 2021 shape',sisu_2021_clean.shape)
print (sisu_2021_pib_clean.columns)

sisu_2021_pib_clean shape (2142711, 18)
sisu 2021 shape (2357524, 15)
Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf', 'pib_pc'],
      dtype='object')


In [357]:
sisu_2021_pib_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
municipio              0
ST_APROVADO            0
idade                  0
ano                    0
uf                     0
pib_pc                 0
dtype: int64

In [325]:
sisu_2021_pib_clean.to_parquet('sisu_2021_pib_clean.parquet')

# 2022 pre processing

## Selecionando colunas, concat e to_parquet

In [128]:
colunas = ['NU_ANO','SG_UF_IES','SG_UF_CAMPUS','NO_MUNUCIPIO_CAMPUS', 'NO_CURSO',
                                   'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA','NU_CPF', 'CO_INSCRICAO_ENEM',
                                   'NO_INSCRITO', 'TP_SEXO','DT_NASCIMENTO', 'SG_UF_CANDIDATO', 
                                   'MUNICIPIO_CANDIDATO','ST_APROVADO']

sisu_regular_1_2022 = pd.read_csv ('ListagemChamadaRegular_2022.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_regular_2_2022 = pd.read_csv ('ListagemChamadaRegular_2022.csv', sep= ';',usecols = colunas,low_memory=False)
sisu_espera_1_2022 = pd.read_csv ('ListagemListaEspera_2021-1.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)
sisu_espera_2_2022 = pd.read_csv ('ListagemListaEspera_2021-2.csv', sep = '|', encoding = 'latin1',usecols = colunas,low_memory=False)

sisu_concat_2022 = pd.concat ([sisu_regular_1_2022,sisu_regular_2_2022,sisu_espera_1_2022,sisu_espera_2_2022])

sisu_concat_2022.to_parquet ('sisu_2022_clean.parquet')


sisu_2022_clean = pd.read_parquet ('sisu_2022_clean.parquet')


In [309]:

sisu_2022_clean.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'MUNICIPIO_CANDIDATO', 'ST_APROVADO'],
      dtype='object')

## Drop and missing values

In [311]:
sisu_2022_clean.dropna(inplace = True)
sisu_2022_clean.drop_duplicates()
sisu_2022_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
MUNICIPIO_CANDIDATO    0
ST_APROVADO            0
dtype: int64

## date dealing


In [313]:
df22_all = sisu_2022_clean.copy()


# Esta função converte a data de nascimento para idade
def age(born):
    # Se 'born' já for um Timestamp, não precisa de conversão
    if isinstance(born, str):
        born = datetime.strptime(born, "%Y/%m/%d").date()
    elif isinstance(born, pd.Timestamp):
        born = born.date()
    
    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df22_all['idade'] = df22_all['DT_NASCIMENTO'].apply(age)
df22_all.columns

Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'MUNICIPIO_CANDIDATO', 'ST_APROVADO', 'idade'],
      dtype='object')

## 2021 merge PIB 

- o dataframe 'pib_pc_clean_municipio_pib_2020_edicao.csv' contém o PIB per capita das cidades brasileiras.

- Essa estatísticas refere-se ao ano de 2020;

- O dataframe foi previamente tratado, mantendo somente as *features* relevantes.

In [135]:

pibpc = pd.read_csv ('pib_pc_clean_municipio_pib_2020_edicao.csv')

print ('shape: ', pibpc.shape)
pibpc.head(30)

shape:  (5570, 4)


,ano,uf,municipio,pib_pc
0,2020,RO,alta floresta doeste,25091
1,2020,RO,ariquemes,25730
2,2020,RO,cabixi,32226
3,2020,RO,cacoal,29331
4,2020,RO,cerejeiras,37069
5,2020,RO,colorado do oeste,23605
6,2020,RO,corumbiara,37171
7,2020,RO,costa marques,13936
8,2020,RO,espigão doeste,20380
9,2020,RO,guajará-mirim,21148


In [315]:
sisu_2022_clean = df22_all.copy()
print ('shape: ',sisu_2022_clean.shape)

shape:  (1986049, 15)


In [317]:


sisu_2022_clean.rename (columns = {'NO_MUNUCIPIO_CAMPUS': 'NO_MUNICIPIO_CAMPUS','MUNICIPIO_CANDIDATO':'municipio'}, inplace = True)

sisu_2022_clean ['municipio'] = sisu_2022_clean.municipio.str.lower()

sisu_2022_pib = pd.merge (sisu_2022_clean, pibpc, how = 'inner', on = 'municipio')
sisu_2022_pib_clean = sisu_2022_pib.drop_duplicates (subset = ['NO_INSCRITO','NO_CURSO','NU_CPF','DT_NASCIMENTO', 'municipio', 'ST_APROVADO'])
sisu_2022_pib_clean = sisu_2022_pib_clean.dropna()

print ('sisu_2022_pib_clean shape',sisu_2022_pib_clean.shape)
print ('sisu 2022 shape',sisu_2022_clean.shape)
print (sisu_2022_pib_clean.columns)

sisu_2022_pib_clean shape (1809271, 18)
sisu 2022 shape (1986049, 15)
Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO', 'SG_UF_CANDIDATO',
       'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf', 'pib_pc'],
      dtype='object')


In [355]:
sisu_2022_pib_clean.isnull().sum()

NU_ANO                 0
SG_UF_IES              0
SG_UF_CAMPUS           0
NO_MUNICIPIO_CAMPUS    0
NO_CURSO               0
TP_MOD_CONCORRENCIA    0
DS_MOD_CONCORRENCIA    0
NU_CPF                 0
NO_INSCRITO            0
TP_SEXO                0
DT_NASCIMENTO          0
SG_UF_CANDIDATO        0
municipio              0
ST_APROVADO            0
idade                  0
ano                    0
uf                     0
pib_pc                 0
dtype: int64

In [323]:
sisu_2022_pib_clean.to_parquet('sisu_2022_pib_clean.parquet')

# Concat All Dataframes

In [388]:
df_all_concat = pd.concat ([sisu_2018_pib_clean,sisu_2019_pib_clean,sisu_2020_pib_clean,sisu_2021_pib_clean,sisu_2022_pib_clean])

print ('Shape: ', df_all_concat.shape)
print ('Columns: ',df_all_concat.columns)


Shape:  (11630219, 19)
Columns:  Index(['NU_ANO', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'NO_CURSO', 'TP_MOD_CONCORRENCIA', 'DS_MOD_CONCORRENCIA', 'NU_CPF',
       'CO_INSCRICAO_ENEM', 'NO_INSCRITO', 'TP_SEXO', 'DT_NASCIMENTO',
       'SG_UF_CANDIDATO', 'municipio', 'ST_APROVADO', 'idade', 'ano', 'uf',
       'pib_pc'],
      dtype='object')


In [399]:
df_all_concat.to_parquet('df_17_22_concat.parquet')
df_all_concat.head()

,ano_concorr,SG_UF_IES,SG_UF_CAMPUS,NO_MUNICIPIO_CAMPUS,curso,tipo_concorr,modo_concorr,cpf,CO_INSCRICAO_ENEM,nome,sexo,nascimento,uf_cand,municipio,aprovado,idade,ano_pib,uf_pib,pib_pc
0,2018,RS,RS,Rio Grande,engenharia de alimentos,L,"Candidatos que, independentemente da renda (ar...",XXX.373.970-XX,17XXXXXXXX66,DOUGLAS HENRIQUE DOS SANTOS BOECK,M,1995-12-17,RS,estância velha,N,29,2020,RS,30444
1,2018,RS,RS,Santa Maria,engenharia aeroespacial,A,Ampla concorrência,XXX.477.610-XX,17XXXXXXXX58,FELIPE BRANCO,M,2000-04-05,RS,estância velha,N,24,2020,RS,30444
2,2018,RS,RS,Porto Alegre,arquitetura e urbanismo,L,Candidatos com renda familiar bruta per capita...,XXX.165.600-XX,17XXXXXXXX36,AUGUSTO MATHIAS KLAUCK,M,1999-06-23,RS,estância velha,N,25,2020,RS,30444
3,2018,RS,RS,Porto Alegre,engenharia ambiental,A,Ampla concorrência,XXX.711.620-XX,17XXXXXXXX74,KATILCE BOKORNY POHREN,F,1993-04-07,RS,estância velha,N,31,2020,RS,30444
4,2018,RJ,RJ,Rio de Janeiro,biomedicina,A,Ampla concorrência,XXX.560.340-XX,17XXXXXXXX30,MAINARA DA ROSA,F,2000-06-07,RS,estância velha,N,24,2020,RS,30444


In [397]:
df_all_concat.isnull().sum()

ano_concorr                  0
SG_UF_IES                    0
SG_UF_CAMPUS                 0
NO_MUNICIPIO_CAMPUS          0
curso                        0
tipo_concorr                 0
modo_concorr                 0
cpf                          0
CO_INSCRICAO_ENEM      3951982
nome                         0
sexo                         0
nascimento                   0
uf_cand                      0
municipio                    0
aprovado                     0
idade                        0
ano_pib                      0
uf_pib                       0
pib_pc                       0
dtype: int64

# Agrupando por ocupações

-  Os cursos foram transformados em ocupaçẽos, segundo o Catálogo Brasileiro de Ocupações (CBO);
-  Muitas formações podem ser atribuídas à uma ocupação e vice-versa;
-  O tratamento e agrupamento de ocupações está no notebook: algoritmo_cursos_to_ocupacoes.ipynb

In [122]:
# df resultante

df_17_22_pib_ocupacoes = pd.read_parquet('df_17_22_pib_ocupacoes.parquet')
print ('columns: ',df_17_22_pib_ocupacoes.columns)
df_17_22_pib_ocupacoes.head()

columns:  Index(['ano_concorr', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'curso', 'tipo_concorr', 'modo_concorr', 'cpf', 'nome', 'sexo',
       'nascimento', 'uf_cand', 'municipio', 'aprovado', 'idade', 'ano_pib',
       'uf_pib', 'pib_pc', 'ocupacoes'],
      dtype='object')


,ano_concorr,SG_UF_IES,SG_UF_CAMPUS,NO_MUNICIPIO_CAMPUS,curso,tipo_concorr,modo_concorr,cpf,nome,sexo,nascimento,uf_cand,municipio,aprovado,idade,ano_pib,uf_pib,pib_pc,ocupacoes
0,2018,RS,RS,Rio Grande,engenharia de alimentos,L,"Candidatos que, independentemente da renda (ar...",XXX.373.970-XX,DOUGLAS HENRIQUE DOS SANTOS BOECK,M,1995-12-17,RS,estância velha,N,29,2020,RS,30444,engenheiro de produção
1,2018,RS,RS,Santa Maria,engenharia aeroespacial,A,Ampla concorrência,XXX.477.610-XX,FELIPE BRANCO,M,2000-04-05,RS,estância velha,N,24,2020,RS,30444,engenheiro aeronáutico
2,2018,RS,RS,Porto Alegre,arquitetura e urbanismo,L,Candidatos com renda familiar bruta per capita...,XXX.165.600-XX,AUGUSTO MATHIAS KLAUCK,M,1999-06-23,RS,estância velha,N,25,2020,RS,30444,arquiteto
3,2018,RS,RS,Porto Alegre,engenharia ambiental,A,Ampla concorrência,XXX.711.620-XX,KATILCE BOKORNY POHREN,F,1993-04-07,RS,estância velha,N,31,2020,RS,30444,engenheiro ambiental
4,2018,RJ,RJ,Rio de Janeiro,biomedicina,A,Ampla concorrência,XXX.560.340-XX,MAINARA DA ROSA,F,2000-06-07,RS,estância velha,N,24,2020,RS,30444,biomédico


# Base salarial por ocupações

- optei por deixar os daods de salários em dtype == int

In [123]:
df_sal = pd.read_csv ('base_salarial_22_23.csv')
df_sal.head()

,cbo,cargo,carga_horaria,piso_22,media_22,mediana_22,teto_22,hora_22,piso_23,media_23,mediana_23,teto_23,hora_23
0,231310,professor de artes do ensino fundamental,28,3032,3117,2400,7178,22,"3.143,89","3.232,16","2.536,00","7.531,20","23,12"
1,262705,Músico Interprete Cantor,40,2401,2468,2100,5129,12,"2.362,16","2.428,49","2.046,00","5.113,78","12,35"
2,262615,Músico Regente,33,2494,2564,2200,5359,15,"2.547,18","2.618,70","2.200,00","5.511,91","16,09"
3,226305,Musicoterapeuta,30,2903,2985,2757,5744,19,"2.864,90","2.945,34","2.700,00","5.705,96","19,79"
4,262620,Musicólogo,33,2189,2251,2000,4366,13,"2.164,78","2.225,56","2.000,00","4.277,71","13,21"


In [10]:
df_sal.columns

Index(['cbo', 'cargo', 'carga_horaria', 'piso_22', 'media_22', 'mediana_22',
       'teto_22', 'hora_22', 'piso_23', 'media_23', 'mediana_23', 'teto_23',
       'hora_23'],
      dtype='object')

## Pre processamento: lower(), strip(), float, negative values

In [124]:
# Rename coluna CBO

df = df_sal.rename (columns = {'cargo': 'ocupacoes'})



# Tratando a coluna 'ocupacoes'

df ['ocupacoes'] = df.ocupacoes.astype(pd.StringDtype())

df['ocupacoes'] = df['ocupacoes'].fillna('').str.lower().str.strip()




df.head()


,cbo,ocupacoes,carga_horaria,piso_22,media_22,mediana_22,teto_22,hora_22,piso_23,media_23,mediana_23,teto_23,hora_23
0,231310,professor de artes do ensino fundamental,28,3032,3117,2400,7178,22,"3.143,89","3.232,16","2.536,00","7.531,20","23,12"
1,262705,músico interprete cantor,40,2401,2468,2100,5129,12,"2.362,16","2.428,49","2.046,00","5.113,78","12,35"
2,262615,músico regente,33,2494,2564,2200,5359,15,"2.547,18","2.618,70","2.200,00","5.511,91","16,09"
3,226305,musicoterapeuta,30,2903,2985,2757,5744,19,"2.864,90","2.945,34","2.700,00","5.705,96","19,79"
4,262620,musicólogo,33,2189,2251,2000,4366,13,"2.164,78","2.225,56","2.000,00","4.277,71","13,21"


In [125]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cbo            129 non-null    int64 
 1   ocupacoes      129 non-null    string
 2   carga_horaria  129 non-null    int64 
 3   piso_22        129 non-null    int64 
 4   media_22       129 non-null    int64 
 5   mediana_22     129 non-null    int64 
 6   teto_22        129 non-null    int64 
 7   hora_22        129 non-null    int64 
 8   piso_23        129 non-null    object
 9   media_23       129 non-null    object
 10  mediana_23     129 non-null    object
 11  teto_23        129 non-null    object
 12  hora_23        129 non-null    object
dtypes: int64(7), object(5), string(1)
memory usage: 13.2+ KB


In [22]:
df.columns

Index(['cbo', 'ocupacoes', 'carga_horaria', 'piso_22', 'media_22',
       'mediana_22', 'teto_22', 'hora_22', 'piso_23', 'media_23', 'mediana_23',
       'teto_23', 'hora_23'],
      dtype='object')

In [126]:
# tratando ','

x = df.copy()
x.drop(columns = ['hora_22','hora_23'], axis = 1, inplace = True)

for col in x.columns:
    if x[col].dtype == 'object':  
        x[col] = x[col].str[:-4]  
        x[col] = x[col].str.replace('.','')
        #x[col] = x[col].astype(int)

x['hora_22'] = df['hora_22']
x['hora_23'] = df['hora_23']


x.head()

,cbo,ocupacoes,carga_horaria,piso_22,media_22,mediana_22,teto_22,piso_23,media_23,mediana_23,teto_23,hora_22,hora_23
0,231310,professor de artes do ensino fundamental,28,3032,3117,2400,7178,3143,3232,2536,7531,22,"23,12"
1,262705,músico interprete cantor,40,2401,2468,2100,5129,2362,2428,2046,5113,12,"12,35"
2,262615,músico regente,33,2494,2564,2200,5359,2547,2618,2200,5511,15,"16,09"
3,226305,musicoterapeuta,30,2903,2985,2757,5744,2864,2945,2700,5705,19,"19,79"
4,262620,musicólogo,33,2189,2251,2000,4366,2164,2225,2000,4277,13,"13,21"


In [28]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cbo            129 non-null    int64 
 1   ocupacoes      129 non-null    string
 2   carga_horaria  129 non-null    int64 
 3   piso_22        129 non-null    int64 
 4   media_22       129 non-null    int64 
 5   mediana_22     129 non-null    int64 
 6   teto_22        129 non-null    int64 
 7   piso_23        129 non-null    object
 8   media_23       129 non-null    object
 9   mediana_23     129 non-null    object
 10  teto_23        129 non-null    object
 11  hora_22        129 non-null    int64 
 12  hora_23        129 non-null    object
dtypes: int64(7), object(5), string(1)
memory usage: 13.2+ KB


In [132]:
# Object to 'int'
x = x.dropna()

for col in x.columns:
    if x[col].dtype == 'object': 
        x[col] = x[col].str.strip() 
        if x[col].str.isdigit().all():  
            x[col] = x[col].astype(int)
        else:
            print(f"A coluna '{col}' contém valores não numéricos e não pode ser convertida para int.")
print ('*'*40)
x.info()


A coluna 'mediana_23' contém valores não numéricos e não pode ser convertida para int.
A coluna 'hora_23' contém valores não numéricos e não pode ser convertida para int.
****************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cbo            129 non-null    int64 
 1   ocupacoes      129 non-null    string
 2   carga_horaria  129 non-null    int64 
 3   piso_22        129 non-null    int64 
 4   media_22       129 non-null    int64 
 5   mediana_22     129 non-null    int64 
 6   teto_22        129 non-null    int64 
 7   piso_23        129 non-null    int64 
 8   media_23       129 non-null    int64 
 9   mediana_23     129 non-null    object
 10  teto_23        129 non-null    int64 
 11  hora_22        129 non-null    int64 
 12  hora_23        129 non-null    object
dtypes: int64(10), object(2), string(1

In [63]:
x.isnull().sum()

cbo              0
ocupacoes        0
carga_horaria    0
piso_sal         0
media_sal        0
mediana_sal      0
teto_sal         0
sal_hora         0
dtype: int64

In [88]:
x.head()

,cbo,ocupacoes,carga_horaria,piso_22,media_22,mediana_22,teto_22,piso_23,media_23,mediana_23,teto_23,hora_22,hora_23
0,231310,professor de artes do ensino fundamental,28,3032,3117,2400,7178,3143,3232,2536,7531,22,"23,12"
1,262705,músico interprete cantor,40,2401,2468,2100,5129,2362,2428,2046,5113,12,"12,35"
2,262615,músico regente,33,2494,2564,2200,5359,2547,2618,2200,5511,15,"16,09"
3,226305,musicoterapeuta,30,2903,2985,2757,5744,2864,2945,2700,5705,19,"19,79"
4,262620,musicólogo,33,2189,2251,2000,4366,2164,2225,2000,4277,13,"13,21"


In [98]:

# convertento 'object' para 'int'
x['hora_23'] = x['hora_23'].str.replace(',', '.')
x['hora_23'] = pd.to_numeric(x['hora_23'], errors='coerce')
x = x.dropna(subset=['hora_23'])
x['hora_23'] = x['hora_23'].astype(int)


# convertento 'mediana_23' par a'int'
x['mediana_23'] = x['mediana_23'].replace('', pd.NA)
x['mediana_23'] = pd.to_numeric(x['mediana_23'], errors='coerce')
x = x.dropna(subset=['mediana_23'])  # Remove linhas com NaN
x['mediana_23'] = x['mediana_23'].astype(int)


df_sal_clean = x.copy()
df_sal_clean.head()


,cbo,ocupacoes,carga_horaria,piso_22,media_22,mediana_22,teto_22,piso_23,media_23,mediana_23,teto_23,hora_22,hora_23
0,231310,professor de artes do ensino fundamental,28,3032,3117,2400,7178,3143,3232,2536,7531,22,23
1,262705,músico interprete cantor,40,2401,2468,2100,5129,2362,2428,2046,5113,12,12
2,262615,músico regente,33,2494,2564,2200,5359,2547,2618,2200,5511,15,16
3,226305,musicoterapeuta,30,2903,2985,2757,5744,2864,2945,2700,5705,19,19
4,262620,musicólogo,33,2189,2251,2000,4366,2164,2225,2000,4277,13,13


In [101]:
df_sal_clean.to_csv ('df_salario_22_23_clean.csv', index = False)

## Merge entre bases sisu + ocupações + pib e salários das ocupações

In [103]:
# dataframe sisu e salarial

df_17_22_pib_ocupacoes = pd.read_parquet('df_17_22_pib_ocupacoes.parquet')
df_sal_clean = pd.read_csv ('df_salario_22_23_clean.csv')

df_17_22_pib_ocupacoes.head()

,ano_concorr,SG_UF_IES,SG_UF_CAMPUS,NO_MUNICIPIO_CAMPUS,curso,tipo_concorr,modo_concorr,cpf,nome,sexo,nascimento,uf_cand,municipio,aprovado,idade,ano_pib,uf_pib,pib_pc,ocupacoes
0,2018,RS,RS,Rio Grande,engenharia de alimentos,L,"Candidatos que, independentemente da renda (ar...",XXX.373.970-XX,DOUGLAS HENRIQUE DOS SANTOS BOECK,M,1995-12-17,RS,estância velha,N,29,2020,RS,30444,engenheiro de produção
1,2018,RS,RS,Santa Maria,engenharia aeroespacial,A,Ampla concorrência,XXX.477.610-XX,FELIPE BRANCO,M,2000-04-05,RS,estância velha,N,24,2020,RS,30444,engenheiro aeronáutico
2,2018,RS,RS,Porto Alegre,arquitetura e urbanismo,L,Candidatos com renda familiar bruta per capita...,XXX.165.600-XX,AUGUSTO MATHIAS KLAUCK,M,1999-06-23,RS,estância velha,N,25,2020,RS,30444,arquiteto
3,2018,RS,RS,Porto Alegre,engenharia ambiental,A,Ampla concorrência,XXX.711.620-XX,KATILCE BOKORNY POHREN,F,1993-04-07,RS,estância velha,N,31,2020,RS,30444,engenheiro ambiental
4,2018,RJ,RJ,Rio de Janeiro,biomedicina,A,Ampla concorrência,XXX.560.340-XX,MAINARA DA ROSA,F,2000-06-07,RS,estância velha,N,24,2020,RS,30444,biomédico


In [107]:
df_17_22_pib_ocupacoes.columns

Index(['ano_concorr', 'SG_UF_IES', 'SG_UF_CAMPUS', 'NO_MUNICIPIO_CAMPUS',
       'curso', 'tipo_concorr', 'modo_concorr', 'cpf', 'nome', 'sexo',
       'nascimento', 'uf_cand', 'municipio', 'aprovado', 'idade', 'ano_pib',
       'uf_pib', 'pib_pc', 'ocupacoes'],
      dtype='object')

In [109]:
# merge, rename, drop, object to string, negative values

df_17_22_pib_ocupacoes_sal = pd.merge (df_17_22_pib_ocupacoes, df_sal_clean, how = 'left', on = 'ocupacoes')

df_17_22_pib_ocupacoes_sal.dropna(inplace= True)

df_17_22_pib_ocupacoes_sal.drop (columns = ['SG_UF_IES','curso'], axis = 1, inplace = True)

df_17_22_pib_ocupacoes_sal = df_17_22_pib_ocupacoes_sal[df_17_22_pib_ocupacoes_sal['idade'] >= 0]

df_17_22_pib_ocupacoes_sal.rename (columns = {'municipio': 'municipio_cand','SG_UF_CAMPUS': 'uf_campus', 'NO_MUNICIPIO_CAMPUS':'municipio_campus'}, inplace = True)

for col in df_17_22_pib_ocupacoes_sal.columns:
    if df_17_22_pib_ocupacoes_sal[col].dtype == 'object':
        df_17_22_pib_ocupacoes_sal[col] = df_17_22_pib_ocupacoes_sal[col].astype(pd.StringDtype())
    if df_17_22_pib_ocupacoes_sal[col].dtype == 'string':
        df_17_22_pib_ocupacoes_sal[col] = df_17_22_pib_ocupacoes_sal[col].str.lower()

   

print ('*'*40)
print ('DF sisu+pib+ocupações+salários shape: ', df_17_22_pib_ocupacoes_sal.shape)
print ('*'*40)
print ('Colunas: ',df_17_22_pib_ocupacoes_sal.columns)
print ('*'*40)
print ('DF dtypes: ', df_17_22_pib_ocupacoes_sal.info())
print ('*'*40)
df_17_22_pib_ocupacoes_sal.head()

****************************************
DF sisu+pib+ocupações+salários shape:  (10742770, 29)
****************************************
Colunas:  Index(['ano_concorr', 'uf_campus', 'municipio_campus', 'tipo_concorr',
       'modo_concorr', 'cpf', 'nome', 'sexo', 'nascimento', 'uf_cand',
       'municipio_cand', 'aprovado', 'idade', 'ano_pib', 'uf_pib', 'pib_pc',
       'ocupacoes', 'cbo', 'carga_horaria', 'piso_22', 'media_22',
       'mediana_22', 'teto_22', 'piso_23', 'media_23', 'mediana_23', 'teto_23',
       'hora_22', 'hora_23'],
      dtype='object')
****************************************
<class 'pandas.core.frame.DataFrame'>
Index: 10742770 entries, 0 to 11630218
Data columns (total 29 columns):
 #   Column            Dtype         
---  ------            -----         
 0   ano_concorr       int64         
 1   uf_campus         string        
 2   municipio_campus  string        
 3   tipo_concorr      string        
 4   modo_concorr      string        
 5   cpf           

,ano_concorr,uf_campus,municipio_campus,tipo_concorr,modo_concorr,cpf,nome,sexo,nascimento,uf_cand,...,piso_22,media_22,mediana_22,teto_22,piso_23,media_23,mediana_23,teto_23,hora_22,hora_23
0,2018,rs,rio grande,l,"candidatos que, independentemente da renda (ar...",xxx.373.970-xx,douglas henrique dos santos boeck,m,1995-12-17,rs,...,8979.0,9231.0,9350.0,18118.0,9136.0,9392.0,9500.0,18435.0,43.0,44.0
1,2018,rs,santa maria,a,ampla concorrência,xxx.477.610-xx,felipe branco,m,2000-04-05,rs,...,13001.0,13366.0,11976.0,25334.0,13269.0,13642.0,11836.0,26346.0,63.0,64.0
3,2018,rs,porto alegre,a,ampla concorrência,xxx.711.620-xx,katilce bokorny pohren,f,1993-04-07,rs,...,7208.0,7411.0,6891.0,16425.0,7407.0,7615.0,7103.0,16853.0,35.0,36.0
4,2018,rj,rio de janeiro,a,ampla concorrência,xxx.560.340-xx,mainara da rosa,f,2000-06-07,rs,...,2724.0,2800.0,2700.0,5019.0,2743.0,2820.0,2753.0,5041.0,13.0,13.0
5,2018,rs,porto alegre,a,ampla concorrência,xxx.965.550-xx,luis evandro timotheo da silva,m,1993-09-20,rs,...,3039.0,3124.0,2805.0,6228.0,3063.0,3149.0,2837.0,6244.0,18.0,18.0


In [111]:
df_17_22_pib_ocupacoes_sal.to_parquet ('df_17_22_pib_ocupacoes_sal.parquet')
